# 01 — Data Curation

Follows the Fourches et al. curation protocol using the author's `utils_curation.py`:
1. Drop inconclusive / missing SMILES rows
2. Remove stereo, salts, organometallics, mixtures
3. Standardize with ChEMBL Structure Pipeline
4. Remove duplicate InChIKeys with discordant labels
5. Label: cytotoxic = 1 if EC50 ≤ 10 µM

Expected output:
- 3T3:   ~66,620 compounds (4,007 cytotoxic)
- HEK293: ~64,094 compounds (6,141 cytotoxic)

In [ ]:

import sys, os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# utils_curation hardcodes paths relative to CWD — must run from reproduction root
os.chdir(os.path.abspath('..'))

sys.path.insert(0, '../author_pipeline/functions')
from utils_curation import curate, group, dupRemovalClassification, removeListedValues

os.makedirs('data/curated', exist_ok=True)
os.makedirs('data/removed_during_curation', exist_ok=True)
os.makedirs('data/data_summary', exist_ok=True)
print('CWD:', os.getcwd())


In [ ]:

def load_pubchem_assay(path):
    """Load PubChem assay CSV.
    Row 0 = real column headers (PUBCHEM_RESULT_TAG, PUBCHEM_SID, ...).
    Rows 1-4 = metadata rows (RESULT_TYPE, RESULT_DESCR, RESULT_UNIT,
    RESULT_IS_ACTIVE_CONCENTRATION, RESULT_ATTR_CONC_MICROMOL) — drop them.
    """
    df = pd.read_csv(path, low_memory=False)
    # Drop the embedded metadata rows; actual data starts where
    # PUBCHEM_RESULT_TAG is a plain integer string
    metadata_tags = {'RESULT_TYPE', 'RESULT_DESCR', 'RESULT_UNIT',
                     'RESULT_IS_ACTIVE_CONCENTRATION', 'RESULT_ATTR_CONC_MICROMOL'}
    df = df[~df['PUBCHEM_RESULT_TAG'].isin(metadata_tags)].reset_index(drop=True)
    return df

def prepare_raw(df, name):
    """Rename PubChem columns to the names expected by utils_curation."""
    print(f'\n=== {name} ===')
    print(f'Rows after metadata removal: {len(df):,}')

    df = df.rename(columns={
        'PUBCHEM_EXT_DATASOURCE_SMILES': 'SMILES',
        'PUBCHEM_ACTIVITY_OUTCOME':      'Outcome',
        'PUBCHEM_CID':                   'PUBCHEM_CID',
    })

    # Author maps Active->1, Inactive->0 and drops everything else (Inconclusive)
    outcome_map = {'Active': 1, 'Inactive': 0}
    df['Outcome'] = df['Outcome'].map(outcome_map)
    df = df.dropna(subset=['Outcome'])
    df['Outcome'] = df['Outcome'].astype(int)

    df = df.dropna(subset=['SMILES'])
    df = df[df['SMILES'].astype(str).str.strip() != '']

    print(f'After dropping inconclusive/missing SMILES: {len(df):,}')
    print(f"  Active (1): {(df['Outcome']==1).sum():,}  Inactive (0): {(df['Outcome']==0).sum():,}")
    return df


In [ ]:

DATASETS = {
    '3T3':    'data/raw/3T3_raw.csv',
    'HEK293': 'data/raw/HEK293_raw.csv',
}

curated_dfs = {}

for name, path in DATASETS.items():
    print(f'\n{"="*50}')
    print(f'Processing {name}')

    df_raw = load_pubchem_assay(path)
    df = prepare_raw(df_raw, name)

    # Step 1: Curation (stereo removal, salt stripping, organometallics,
    #         mixtures, ChEMBL standardization) — uses utils_curation.curate()
    #         which expects columns: SMILES, Outcome
    df_currated = curate(df, 'data/curated/')

    # Step 2: Group by InChIKey, then remove discordant duplicates (std > 0)
    grouped = group(df_currated, ['Outcome', 'SMILES'])
    grouped = dupRemovalClassification(grouped, 'Outcome', name)

    # Step 3: Flatten list-valued cells back to scalar
    grouped = removeListedValues(grouped)

    # Step 4: Restore PUBCHEM_CID by merging on SMILES
    cid_map = df[['SMILES', 'PUBCHEM_CID']].drop_duplicates('SMILES')
    grouped = grouped.merge(cid_map, on='SMILES', how='left')

    print(f'After duplicate removal: {len(grouped):,}')
    print(f"  Cytotoxic    (1): {int((grouped['Outcome']==1).sum()):,}")
    print(f"  Noncytotoxic (0): {int((grouped['Outcome']==0).sum()):,}")

    out_path = f'data/curated/{name}_curated.csv'
    grouped[['SMILES', 'Outcome', 'PUBCHEM_CID']].to_csv(out_path, index=False)
    print(f'Saved: {out_path}')

    curated_dfs[name] = grouped


In [ ]:

# Compare against author's reference CSVs
expected = {
    '3T3':    {'total': 66620, 'cytotoxic': 4007,  'noncytotoxic': 62613},
    'HEK293': {'total': 64094, 'cytotoxic': 6141,  'noncytotoxic': 57953},
}

author_paths = {
    '3T3':    '../author_cytosafe/Datasets/3T3_curated.csv',
    'HEK293': '../author_cytosafe/Datasets/HEK_curated.csv',
}

for name, df in curated_dfs.items():
    n_cyto = int((df['Outcome'] == 1).sum())
    n_non  = int((df['Outcome'] == 0).sum())
    n_tot  = len(df)
    exp    = expected[name]

    match_total  = '✓' if n_tot  == exp['total']      else f'✗ (off by {n_tot - exp["total"]})'
    match_cyto   = '✓' if n_cyto == exp['cytotoxic']  else f'✗ (off by {n_cyto - exp["cytotoxic"]})'
    match_non    = '✓' if n_non  == exp['noncytotoxic'] else f'✗ (off by {n_non - exp["noncytotoxic"]})'

    print(f'{name}:')
    print(f'  total={n_tot:,}  expected={exp["total"]:,}  {match_total}')
    print(f'  cytotoxic={n_cyto:,}  expected={exp["cytotoxic"]:,}  {match_cyto}')
    print(f'  noncytotoxic={n_non:,}  expected={exp["noncytotoxic"]:,}  {match_non}')

    # SMILES overlap with author dataset
    auth_df = pd.read_csv(author_paths[name])
    our_smiles  = set(df['SMILES'].dropna())
    auth_smiles = set(auth_df['SMILES'].dropna())
    overlap = len(our_smiles & auth_smiles)
    print(f'  SMILES overlap with author CSV: {overlap:,} / {len(auth_smiles):,} ({overlap/len(auth_smiles)*100:.1f}%)')
    print()
